# 01. Mobility factor construction for the 2016-2017 season

This notebook constructs the subway-ridership-based mobility coefficient used in the manuscript. In manuscript notation, the final mobility factor is \(\xi(t)=b(t)\). In the uploaded code, this final column is stored as `gamma`, and the mobility-reduction scenario columns are stored as `gamma_1` through `gamma_5`. No `theta` parameter is used.


In [ ]:
# Repository path setup
# This cell makes the notebook runnable from either the repository root or the notebooks/ directory.
from pathlib import Path
import os


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "notebooks").exists():
            return candidate
    if current.name == "notebooks":
        return current.parent
    return current


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

for directory in [
    "data/metro",
    "data/NHIS/2016~2017",
    "data/mobility_factor/2016~2017",
    "data/Rt/2016~2017",
    "data/Rt/2017~2018",
    "data/Rt/2018~2019",
    "data/Rt/2022~2023",
    "figures/mobility_factor",
    "figures/2016~2017",
    "figures/HeatMap",
    "figures/validation",
    "results/validation",
    "results/tables",
]:
    Path(directory).mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import pandas as pd

df_seoul = pd.read_csv("data/metro/seoul&g_metro.csv")
df_busan = pd.read_csv("data/metro/busan_metro.csv")
df_daejeon = pd.read_csv("data/metro/daejeon_metro.csv")
df_daegu = pd.read_csv("data/metro/daegu_metro.csv")
df_gwangju = pd.read_csv("data/metro/gwangju_metro.csv")

df_seoul['date'] = pd.to_datetime(df_seoul['date'])
df_busan['date'] = pd.to_datetime(df_busan['date'])
df_daejeon['date'] = pd.to_datetime(df_daejeon['date'])
df_daegu['date'] = pd.to_datetime(df_daegu['date'])
df_gwangju['date'] = pd.to_datetime(df_gwangju['date'])

df = pd.concat([df_seoul,df_busan,df_daejeon,df_daegu,df_gwangju], ignore_index=True)
df.sort_values(['date'], inplace=True)
daily = df[df['date'].between('2016-09-01', '2017-08-31')]
daily.info()

In [ ]:
import numpy as np
import pandas as pd

pop_map = {
    "seoul": 22_689_358.5,
    "busan": 3_484_591,
    "daejeon": 1_508_298.5,
    "daegu": 2_479_894,
    "gwangju": 1_466_492
}

target_gus = list(pop_map.keys())

target_daily = (
    daily[daily["city"].isin(target_gus)]
      .groupby(["city", "date"], as_index=False)["people_in"]
      .sum()
      .sort_values(["city", "date"])
)

target_daily["pop"] = target_daily["city"].map(pop_map)

#  m_i(t) = X_i(t) / Pop_i
target_daily["m"] = target_daily["people_in"] / target_daily["pop"]

# 7 Day moving average
target_daily["m_7"] = (
    target_daily
      .groupby(["city"])["m"]
      .transform(lambda s: s.rolling(window=7, center=True, min_periods=1).mean())
)


# L_i = mean_t m_7_i(t)  (지역별 레벨)
L = target_daily.groupby("city")["m_7"].mean()
L_ref = L.mean()

target_daily["L"] = (
    target_daily
      .groupby(["city"])["m_7"]
      .transform("mean")
)

# c_i(t) = m_i(t) / L_i(t)
target_daily["c"] = target_daily["m"] / target_daily["L"]

# c_i(t) = m_i(t) / L_i(t)

target_daily["b"] = (
    target_daily
      .groupby(["city"])["c"]
      .transform(lambda s: s.rolling(window=7, center=True, min_periods=1).mean())
)

target_daily["gamma"] = target_daily["b"]
target_daily["gamma_1"] = (1 - 0.1) * target_daily["b"]
target_daily["gamma_2"] = (1 - 0.2) * target_daily["b"]
target_daily["gamma_3"] = (1 - 0.3) * target_daily["b"]
target_daily["gamma_4"] = (1 - 0.4) * target_daily["b"]
target_daily["gamma_5"] = (1 - 0.5) * target_daily["b"]


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

seoul_data = target_daily[target_daily['city'] == 'seoul']
seoul_data = seoul_data[(seoul_data['date'] >= '2016-09-01') & (seoul_data['date'] <= '2017-08-31')]

plt.figure(figsize=(18, 8))
plt.plot(seoul_data["date"], seoul_data["m"], label="Seoul")

plt.title("m(t))")
plt.xlabel("Date")
plt.ylabel("m(t)")
# plt.axhline(1.0, linestyle="--")
plt.legend()
plt.yticks([0.05, 0.1, 0.15, 0.2, 0.25, 0.30])
ax = plt.gca()
# ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.tight_layout()
plt.grid(True,which='major',linestyle='-',linewidth=0.5,color='0.85')
plt.savefig("figures/mobility_factor/Seoul_m.eps",format='eps',dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

seoul_data = target_daily[target_daily['city'] == 'seoul']
seoul_data = seoul_data[(seoul_data['date'] >= '2016-09-01') & (seoul_data['date'] <= '2017-08-31')]

plt.figure(figsize=(18, 8))
plt.plot(seoul_data["date"], seoul_data["c"], label="Seoul")

plt.title("c(t))")
plt.xlabel("Date")
plt.ylabel("c(t)")
plt.axhline(1.0, linestyle="--", color='r')
plt.legend()
plt.yticks([0.4, 0.6, 0.8, 1.0, 1.2, 1.4])
ax = plt.gca()
# ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.tight_layout()
plt.grid(True,which='major',linestyle='-',linewidth=0.5,color='0.85')
plt.savefig("figures/mobility_factor/Seoul_c.eps",format='eps',dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

seoul_data = target_daily[target_daily['city'] == 'seoul']
seoul_data = seoul_data[(seoul_data['date'] >= '2016-09-01') & (seoul_data['date'] <= '2017-08-31')]

plt.figure(figsize=(18, 8))
plt.plot(seoul_data["date"], seoul_data["b"], label="Seoul")

plt.title("b(t))")
plt.xlabel("Date")
plt.ylabel("b(t)")
plt.axhline(1.0, linestyle="--", color='r')
plt.yticks([0.6, 0.8, 1.0, 1.2])
plt.legend()

ax = plt.gca()
# ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.tight_layout()
plt.grid(True,which='major',linestyle='-',linewidth=0.5,color='0.85')
plt.savefig("figures/mobility_factor/Seoul_b.eps",format='eps',dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

seoul_data = target_daily[target_daily['city'] == 'busan']
seoul_data = seoul_data[(seoul_data['date'] >= '2016-09-01') & (seoul_data['date'] <= '2017-08-31')]

plt.figure(figsize=(10, 4))
plt.plot(seoul_data["date"], seoul_data["gamma"], label="Seoul")
plt.plot(seoul_data["date"], seoul_data["gamma_1"], label="c = 0.1")
plt.plot(seoul_data["date"], seoul_data["gamma_2"], label="c = 0.2")
plt.plot(seoul_data["date"], seoul_data["gamma_3"], label="c = 0.3")
plt.plot(seoul_data["date"], seoul_data["gamma_4"], label="c = 0.4")
plt.plot(seoul_data["date"], seoul_data["gamma_5"], label="c = 0.5")


plt.title("$b_i$(t))")
plt.xlabel("Date")
plt.ylabel("$b_i$(t)")
plt.axhline(1.0, linestyle="--")
plt.legend()

ax = plt.gca()
# ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.tight_layout()
plt.show()

In [ ]:
# 이동률 저장
seoul_data = target_daily[target_daily['city'] == 'seoul']
busan_data = target_daily[target_daily['city'] == 'busan']
daegu_data = target_daily[target_daily['city'] == 'daegu']
daejeon_data = target_daily[target_daily['city'] == 'daejeon']
gwangju_data = target_daily[target_daily['city'] == 'gwangju']
seoul_data.to_csv('data/mobility_factor/2016~2017/Seoul_gamma.csv')
busan_data.to_csv('data/mobility_factor/2016~2017/Busan_gamma.csv')
daegu_data.to_csv('data/mobility_factor/2016~2017/Daegu_gamma.csv')
daejeon_data.to_csv('data/mobility_factor/2016~2017/Daejeon_gamma.csv')
gwangju_data.to_csv('data/mobility_factor/2016~2017/Gwangju_gamma.csv')